# Agent Evaluation — Tool Selection + Answer Quality

Two evaluations specific to an agentic system.

## 1. Tool-selection correctness (new failure mode)

Picking the wrong tool is distinct from retrieval or generation quality, so
it gets its own metric and test set (`data/tool-selection-testset.csv`, 15
questions with expected tool calls). We check whether the agent's chosen
tools match the expected set. Requires `OPENAI_API_KEY`.

## 2. Answer quality — 3-label LLM-as-judge

Same categorical scheme as the rest of the portfolio, over the agent's final
answers.

In [ ]:
import sys, json
sys.path.append("..")
import pandas as pd
from pv_assistant.agent import run_agent

## Tool-selection correctness

In [ ]:
tests = pd.read_csv("../data/tool-selection-testset.csv")

def expected_set(s):
    return set(s.split(";"))

rows = []
for t in tests.itertuples():
    result = run_agent(t.question)
    got = set(result["tools_used"])
    exp = expected_set(t.expected_tools)
    rows.append({
        "question": t.question,
        "expected": ";".join(sorted(exp)),
        "got": ";".join(sorted(got)) or "(none)",
        "exact_match": got == exp,
        "covers_expected": exp.issubset(got),  # called at least the right tools
    })

tool_df = pd.DataFrame(rows)
print(f"exact tool-set match:   {tool_df.exact_match.mean():.0%}")
print(f"called all needed tools: {tool_df.covers_expected.mean():.0%}")
tool_df

`exact_match` is strict (no extra tools); `covers_expected` allows the
agent to call additional tools as long as it called the right ones. A pure
regulation question triggering an unnecessary AEMS call fails `exact_match`
but passes `covers_expected` — report both, they mean different things.

## Answer quality — LLM-as-judge over agent answers

In [ ]:
SAMPLE = tests.question.tolist()  # or a larger held-out set
qa = []
for q in SAMPLE:
    r = run_agent(q)
    qa.append({"question": q, "verdict": r["verdict"],
               "relevance": r["relevance"], "tools": ";".join(r["tools_used"]),
               "cost": r["openai_cost"]})
qa_df = pd.DataFrame(qa)
qa_df.relevance.value_counts(normalize=True)

In [ ]:
tool_df.to_csv("../data/tool-selection-results.csv", index=False)
qa_df.to_csv("../data/agent-answer-eval.csv", index=False)

## Sanity-check the judge

As everywhere in the portfolio: hand-review a sample of judge labels. The
judge is a scalable proxy for regression-catching and A/B comparison, not
ground truth — and in this safety-critical domain that caveat carries extra
weight.